In [1]:
import pandas as pd
import numpy as np

In [2]:
temp_df = pd.read_csv("..\\Datasets\\IMDB_Dataset_full.csv")
temp_df.shape

(50000, 2)

In [3]:
df = temp_df.iloc[:10000]
df.shape

(10000, 2)

In [4]:
df.drop_duplicates(inplace=True)

C:\Users\Acer1\AppData\Local\Temp\ipykernel_19976\3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [5]:
import re
def remove_html_tags(raw_text):
  cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
  return cleaned_text

In [ ]:
df['review'] = df['review'].apply(remove_html_tags)

C:\Users\Acer1\AppData\Local\Temp\ipykernel_19976\3667178324.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_html_tags)


In [7]:
df['review'] = df['review'].apply(lambda x: x.lower())

C:\Users\Acer1\AppData\Local\Temp\ipykernel_19976\2432328559.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: x.lower())


In [8]:
from nltk.corpus import stopwords

In [9]:
stop_words = stopwords.words('english')
stop_words[:10]

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']

In [10]:
def remove_stop_words(text):
  new_text = []
  for word in text.split():
    if word not in stop_words:
      new_text.append(word)
  return ' '.join(new_text)

In [11]:
df['review'] = df['review'].apply(remove_stop_words)

C:\Users\Acer1\AppData\Local\Temp\ipykernel_19976\2082863007.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_stop_words)


In [14]:
import gensim
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

In [15]:
story = []
for doc in df['review']:
  raw_sent = sent_tokenize(doc)
  for sent in raw_sent:
    story.append(simple_preprocess(sent))

In [18]:
model = gensim.models.Word2Vec(window=10, min_count=2)

In [20]:
model.build_vocab(story)

In [21]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5850081, 6186875)

In [22]:
len(model.wv.index_to_key)

31845

In [24]:
def document_vector(doc):
  doc = [word for word in doc.split() if word in model.wv.index_to_key]
  return np.mean(model.wv[doc], axis=0)

In [25]:
document_vector(df['review'].values[0])

array([-0.0109319 ,  0.378754  ,  0.2422129 , -0.06599218,  0.01190558,
       -0.49584803,  0.1113292 ,  0.8052019 , -0.21582592, -0.23977731,
       -0.17992401, -0.5204955 , -0.06101466,  0.3517286 ,  0.20328233,
       -0.21021248,  0.07592834, -0.5331737 , -0.0385862 , -0.66888386,
        0.05381991,  0.11002259,  0.1642774 , -0.21579435, -0.21301785,
       -0.17200933, -0.13208432, -0.233191  , -0.3961545 , -0.01421406,
        0.3568751 ,  0.08610661, -0.01936102, -0.14208524, -0.45086852,
        0.35366723,  0.00509686, -0.3414199 , -0.18854873, -0.7556711 ,
        0.14673916, -0.2044411 , -0.11817635, -0.04628357,  0.46124583,
       -0.14076063, -0.45624247, -0.1327863 ,  0.08417755,  0.32380807,
        0.13852783, -0.25360125, -0.33094206, -0.1468318 , -0.32722047,
        0.06105632,  0.44433296,  0.19261163, -0.3058587 ,  0.06168508,
       -0.06506694,  0.15081845, -0.13138191,  0.01933226, -0.40287328,
        0.4597715 , -0.03585881,  0.09276054, -0.56724703,  0.34

In [26]:
from tqdm import tqdm

In [27]:
X = []
for doc in tqdm(df['review'].values):
  X.append(document_vector(doc ) )

100%|██████████| 9983/9983 [10:12<00:00, 16.30it/s]


In [28]:
X = np.array(X)

In [29]:
X.shape

(9983, 100)

In [30]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [31]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [33]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)

In [34]:
accuracy_score(y_test,y_pred)

0.7701552328492739